# 05 — PA-RL inference-time evaluation

Standalone скрипт для eval **уже сохранённой** policy + critic с PA-RL **inference**.

**Что делает**:
- Загружает policy из CHECKPOINT_PATH (offline distilled best или другой)
- Загружает critic из CRITIC_PATH
- На каждом env step делает: sample N → global rerank → local refine → execute optimized first action
- Сравнивает plain eval (just `policy.select_action`) и PA-RL inference eval

**Зачем**:
- Paper Section 4.3: "action optimization at both training and inference time"
- Critic-guided action selection даёт +1-2 episode сверху distilled policy
- Не требует training — только eval

**Время на L40S**: 2 × 13 episodes × ~40-100 sec each ≈ **20-45 минут** на full comparison.


## 1. Setup

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MUJOCO_GL"]="egl"; os.environ["PYOPENGL_PLATFORM"]="egl"

import sys, math, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda:0")
torch.backends.cudnn.benchmark = True

# === Пути ===
# Выбери какую policy эвалить. По умолчанию — best offline PA-RL distill checkpoint.
CHECKPOINT_PATH = "/workspace/out/smolvla_parl_online_best_12"
# Альтернативы:
#   "/workspace/out/smolvla_parl_online_best"  — best online
#   "HuggingFaceVLA/smolvla_libero"            — baseline (no distill)

CRITIC_PATH = "/workspace/out/critic_resnet-Copy1.pt"

# === Eval config ===
TASK_DESCRIPTION = "pick up the black bowl between the plate and the ramekin and place it on the plate"
TASK_SUITE_NAME       = "libero_spatial"
TASK_ID               = 0
ENV_IMAGE_SIZE        = 256
SEED                  = 42
MAX_STEPS             = 90
NUM_STABILIZATION_STEPS = 10
INIT_STATES_IDS       = [1, 2, 6, 7, 13, 22, 23, 27, 32, 35, 38, 47, 46]  # canonical eval

# === PA-RL inference hyperparams (paper App B.1) ===
PARL_INFERENCE_N   = 16
PARL_INFERENCE_M   = 4
PARL_INFERENCE_T   = 10
PARL_INFERENCE_ETA = 3e-4
PARL_CHUNK_HORIZON = 8   # Re-generate chunk каждые N env steps для closed-loop (вместо 50 = open loop)
NUM_DENOISE_STEPS  = 10   # для sampling — будет overridden из policy.config

# === Multi-seed evaluation ===
SEEDS = [42, 228, 282, 1488, 1337, 67, 69, 34]   # 3 разных seed для mean ± std (39 episodes total)

# === Какие моды eval запустить ===
RUN_BASELINE_EVAL  = False   # SmolVLA baseline (no distill, no LoRA) — для reference
RUN_PLAIN_EVAL     = True   # distilled policy.select_action (no critic at inference)
RUN_PARL_INFERENCE = False   # distilled + critic-guided PA-RL inference

torch.manual_seed(SEED); np.random.seed(SEED)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Critic:     {CRITIC_PATH}")
print(f"INIT_STATES_IDS: {INIT_STATES_IDS}")
print(f"PA-RL inference: N={PARL_INFERENCE_N}, M={PARL_INFERENCE_M}, T={PARL_INFERENCE_T}, η={PARL_INFERENCE_ETA}")
print(f"Modes: plain={RUN_PLAIN_EVAL}, PA-RL inference={RUN_PARL_INFERENCE}")

from huggingface_hub import login
login(token="", add_to_git_credential=False)


PyTorch: 2.11.0+cu130, CUDA: True
Checkpoint: /workspace/out/smolvla_parl_online_best_12
Critic:     /workspace/out/critic_resnet-Copy1.pt
INIT_STATES_IDS: [1, 2, 6, 7, 13, 22, 23, 27, 32, 35, 38, 47, 46]
PA-RL inference: N=16, M=4, T=10, η=0.0003
Modes: plain=True, PA-RL inference=False


## 2. Загрузка policy (с LoRA если есть)

In [2]:
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy, resize_with_pad, pad_vector, make_att_2d_masks
)
from lerobot.policies.factory import make_pre_post_processors

MODEL_ID_BASE = "HuggingFaceVLA/smolvla_libero"

# === Step 1: Load BASE policy (no LoRA) ===
print(f"Loading BASE policy from {MODEL_ID_BASE}…")
policy = SmolVLAPolicy.from_pretrained(MODEL_ID_BASE).to(device)
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MODEL_ID_BASE,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)
print(f"  base loaded, chunk_size={policy.config.chunk_size}, num_steps={policy.config.num_steps}")

# === Step 2: Apply LoRA wrapping (same как в training) ===
for p in policy.parameters():
    p.requires_grad_(False)


class LoRALinear(nn.Module):
    def __init__(self, base_layer: nn.Linear, rank: int = 32, alpha: float = 8.0):
        super().__init__()
        self.base = base_layer
        for p in self.base.parameters():
            p.requires_grad_(False)
        in_f = base_layer.in_features
        out_f = base_layer.out_features
        self.in_features = in_f
        self.out_features = out_f
        self.lora_A = nn.Linear(in_f, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_f, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)
        self.scaling = alpha / rank
    @property
    def weight(self): return self.base.weight
    @property
    def bias(self):   return self.base.bias
    def forward(self, x):
        base_out = self.base(x)
        x_lora = x.to(self.lora_A.weight.dtype)
        lora_out = self.scaling * self.lora_B(self.lora_A(x_lora))
        return base_out + lora_out.to(base_out.dtype)


def apply_lora(module, target_names, rank=32, alpha=8):
    n_replaced = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and any(t in name for t in target_names):
            setattr(module, name, LoRALinear(child, rank=rank, alpha=alpha))
            n_replaced += 1
        else:
            n_replaced += apply_lora(child, target_names, rank, alpha)
    return n_replaced


LORA_RANK = 32
LORA_ALPHA = 8
target_names = ["q_proj", "k_proj", "v_proj", "o_proj"]

if hasattr(policy.model.vlm_with_expert, "lm_expert"):
    n_lora = apply_lora(policy.model.vlm_with_expert.lm_expert, target_names,
                        rank=LORA_RANK, alpha=LORA_ALPHA)
    print(f"  Applied LoRA to {n_lora} attention projections")
policy.to(device)

# === Step 3: Load checkpoint state dict (only if different from base) ===
if CHECKPOINT_PATH != MODEL_ID_BASE:
    print(f"\nLoading state dict from {CHECKPOINT_PATH}…")
    # Find model weights file in checkpoint dir
    ckpt_dir = CHECKPOINT_PATH
    weight_file = None
    for f in os.listdir(ckpt_dir):
        if f.endswith(".safetensors") or f == "pytorch_model.bin":
            weight_file = os.path.join(ckpt_dir, f)
            break
    
    if weight_file is None:
        raise FileNotFoundError(f"No weight file (.safetensors or pytorch_model.bin) в {ckpt_dir}")
    
    print(f"  weight file: {weight_file}")
    if weight_file.endswith(".safetensors"):
        from safetensors.torch import load_file
        state_dict = load_file(weight_file)
    else:
        state_dict = torch.load(weight_file, map_location=device)
    
    missing, unexpected = policy.load_state_dict(state_dict, strict=False)
    print(f"  Loaded. missing keys: {len(missing)}, unexpected: {len(unexpected)}")
    if len(missing) > 0:
        print(f"    first 3 missing: {missing[:3]}")
    if len(unexpected) > 0:
        print(f"    first 3 unexpected: {unexpected[:3]}")
else:
    print("\n[skipped state dict load] CHECKPOINT_PATH == MODEL_ID_BASE")

policy.eval()
NUM_DENOISE_STEPS = policy.config.num_steps
print(f"\n  chunk_size={policy.config.chunk_size}, num_steps={NUM_DENOISE_STEPS}")
print(f"  total params: {sum(p.numel() for p in policy.parameters())/1e6:.1f}M")
n_lora_params = sum(p.numel() for n, p in policy.named_parameters() if "lora_" in n)
print(f"  LoRA params (loaded): {n_lora_params/1e6:.2f}M")


Loading BASE policy from HuggingFaceVLA/smolvla_libero…


`torch_dtype` is deprecated! Use `dtype` instead!


Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

  base loaded, chunk_size=50, num_steps=10
  Applied LoRA to 128 attention projections

Loading state dict from /workspace/out/smolvla_parl_online_best_12…
  weight file: /workspace/out/smolvla_parl_online_best_12/model.safetensors


  Loaded. missing keys: 0, unexpected: 0

  chunk_size=50, num_steps=10
  total params: 609.4M
  LoRA params (loaded): 4.42M


## 3. Загрузка critic

In [3]:
from torchvision.models import resnet18

CRITIC_IMG_SIZE = 128
RAW_IMG_SIZE    = 256


class ImageEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = resnet18(weights=None)
        self.backbone.fc = nn.Identity()
        self.out_dim = 512
    def forward(self, img):
        return self.backbone(img)


class CriticHead(nn.Module):
    def __init__(self, obs_dim, action_dim=7, hidden=512, action_emb_dim=128):
        super().__init__()
        self.action_emb = nn.Sequential(
            nn.Linear(action_dim, action_emb_dim), nn.LayerNorm(action_emb_dim), nn.ReLU(),
            nn.Linear(action_emb_dim, action_emb_dim), nn.LayerNorm(action_emb_dim), nn.ReLU(),
        )
        self.l1 = nn.Sequential(nn.Linear(obs_dim+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l2 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l3 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l4 = nn.Linear(hidden+action_emb_dim, 1)
    def forward(self, obs, a):
        a_emb = self.action_emb(a)
        x = self.l1(torch.cat([obs, a_emb], -1))
        x = self.l2(torch.cat([x, a_emb], -1))
        x = self.l3(torch.cat([x, a_emb], -1))
        return self.l4(torch.cat([x, a_emb], -1)).squeeze(-1)


class CriticEnsemble(nn.Module):
    def __init__(self, state_dim=8, action_dim=7, hidden=512, state_hidden=64):
        super().__init__()
        self.enc1 = ImageEncoder()
        self.enc2 = ImageEncoder()
        self.state_mlp = nn.Sequential(
            nn.Linear(state_dim, state_hidden), nn.LayerNorm(state_hidden), nn.ReLU(),
            nn.Linear(state_hidden, state_hidden),
        )
        self.obs_dim = 512 + 512 + state_hidden
        self.obs_norm = nn.LayerNorm(self.obs_dim)
        self.q1 = CriticHead(self.obs_dim, action_dim, hidden)
        self.q2 = CriticHead(self.obs_dim, action_dim, hidden)
    def encode(self, img1, img2, state):
        e1 = self.enc1(img1)
        e2 = self.enc2(img2)
        s  = self.state_mlp(state)
        obs = torch.cat([e1, e2, s], dim=-1)
        return self.obs_norm(obs)


critic = CriticEnsemble().to(device)
state = torch.load(CRITIC_PATH, map_location=device, weights_only=False)
critic.load_state_dict(state["critic"])
state_mean_np = state["state_mean"] if isinstance(state["state_mean"], np.ndarray) else state["state_mean"].cpu().numpy()
state_std_np  = state["state_std"]  if isinstance(state["state_std"],  np.ndarray) else state["state_std"].cpu().numpy()
for p in critic.parameters(): p.requires_grad_(False)
critic.eval()
print(f"Critic loaded из {CRITIC_PATH}")
print(f"  Critic params: {sum(p.numel() for p in critic.parameters())/1e6:.1f}M (frozen)")


Critic loaded из /workspace/out/critic_resnet-Copy1.pt
  Critic params: 25.0M (frozen)


## 4. LIBERO env

In [4]:
import builtins
builtins.input = lambda _: "n"
os.environ["LIBERO_DATA_PATH"] = "/workspace/libero_data"
os.makedirs("/workspace/libero_data", exist_ok=True)

from libero.libero import get_libero_path, benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[TASK_SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
task_description_libero = task.language
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

env = OffScreenRenderEnv(bddl_file_name=task_bddl_file,
                         camera_heights=ENV_IMAGE_SIZE, camera_widths=ENV_IMAGE_SIZE)
env.seed(SEED)
init_states = task_suite.get_task_init_states(TASK_ID)
print(f"Task: {task_description_libero}")
print(f"Init states: {len(init_states)} total, eval set: {len(INIT_STATES_IDS)}")


def quat2axisangle(quat):
    quat = quat.astype(np.float32).copy()
    quat[3] = np.clip(quat[3], -1.0, 1.0)
    den = np.sqrt(max(1e-12, 1.0 - quat[3]**2))
    if np.isclose(den, 0.0): return np.zeros(3, dtype=np.float32)
    angle = 2.0 * math.acos(float(quat[3]))
    return ((quat[:3] * angle) / den).astype(np.float32)


def rotate_180(im):
    return np.ascontiguousarray(im[::-1, ::-1])


def downscale_only(img, target_size=CRITIC_IMG_SIZE):
    if target_size != img.shape[-1]:
        return F.interpolate(img, size=target_size, mode='bilinear', align_corners=False)
    return img


# IDENTICAL копия libero_obs_to_lerobot из parl_distill_progressive
def libero_obs_to_lerobot(raw_obs, task_text):
    a_img = rotate_180(raw_obs["agentview_image"])
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"])
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    return {
        "observation.images.image":  torch.from_numpy(a_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.images.image2": torch.from_numpy(w_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.state":         torch.from_numpy(state).unsqueeze(0).float(),
        "task":                      [task_text],
    }


def libero_obs_to_critic_input(raw_obs):
    """(img1_critic, img2_critic, state_norm) для critic.encode (batch=1)."""
    a_img = rotate_180(raw_obs["agentview_image"]).astype(np.float32) / 255.0
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"]).astype(np.float32) / 255.0
    img1_full = torch.from_numpy(a_img).permute(2, 0, 1).unsqueeze(0).to(device)
    img2_full = torch.from_numpy(w_img).permute(2, 0, 1).unsqueeze(0).to(device)
    img1_critic = downscale_only(img1_full)
    img2_critic = downscale_only(img2_full)
    
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    state_norm = (state - state_mean_np) / state_std_np
    state_t = torch.from_numpy(state_norm).unsqueeze(0).to(device)
    return img1_critic, img2_critic, state_t


[robosuite WARNING] No private macro file found! (__init__.py:7)


[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)


[robosuite WARNING] To setup, run: python /venv/main/lib/python3.12/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Local assets not found. Downloading from HuggingFace Hub...
Assets already downloaded at /root/.cache/libero/assets


Task: pick up the black bowl between the plate and the ramekin and place it on the plate
Init states: 50 total, eval set: 13


## 5. PA-RL sampling helper (flow-matching denoise from policy)

In [5]:
@torch.no_grad()
def prepare_policy_inputs(img1_policy, img2_policy, state_raw, batch_size):
    """IDENTICAL копия из parl_distill_progressive.
    
    img1_policy, img2_policy: [B, 3, H, W] float[0,1] images (full-res)
    state_raw: [B, 8] RAW state (not normalized)
    """
    sp = preprocessor({
        "observation.images.image":       torch.zeros(batch_size, 3, 4, 4),
        "observation.images.wrist_image": torch.zeros(batch_size, 3, 4, 4),
        "observation.state":              state_raw.cpu(),
        "task":                           [TASK_DESCRIPTION] * batch_size,
    })
    state_norm  = sp["observation.state"].to(device)
    lang_tokens = sp["observation.language.tokens"].to(device)
    lang_masks  = sp["observation.language.attention_mask"].to(device)
    
    raw = {
        "observation.images.image":            img1_policy,
        "observation.images.image2":           img2_policy,
        "observation.images.wrist_image":      img2_policy,
        "observation.state":                   state_norm,
        "observation.language.tokens":         lang_tokens,
        "observation.language.attention_mask": lang_masks,
    }
    images, img_masks = policy.prepare_images(raw)
    state_p           = policy.prepare_state(raw)
    
    return raw, images, img_masks, state_p, lang_tokens, lang_masks


@torch.no_grad()
def sample_n_actions(images, img_masks, state_p, lang_tokens, lang_masks,
                    n_candidates=None, num_steps_override=None):
    """IDENTICAL копия из parl_distill_progressive — возвращает [B, N, chunk_size, 7]."""
    if n_candidates is None:
        n_candidates = PARL_INFERENCE_N
    
    B = images[0].shape[0] if isinstance(images, list) else images.shape[0]
    
    prefix_embs, prefix_pad_masks, prefix_att_masks = policy.model.embed_prefix(
        images, img_masks, lang_tokens, lang_masks, state=state_p
    )
    prefix_att_2d  = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_pos_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    _, past_kv = policy.model.vlm_with_expert.forward(
        attention_mask=prefix_att_2d, position_ids=prefix_pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, None],
        use_cache=True, fill_kv_cache=True,
    )
    
    actions_shape = (B, policy.config.chunk_size, policy.config.max_action_dim)
    num_steps = num_steps_override if num_steps_override is not None else NUM_DENOISE_STEPS
    dt = -1.0 / num_steps
    
    chunks = []
    for _ in range(n_candidates):
        x_t = policy.model.sample_noise(actions_shape, device)
        for step in range(num_steps):
            t_val = 1.0 + step * dt
            t_tensor = torch.tensor(t_val, device=device).expand(B)
            v_t = policy.model.denoise_step(prefix_pad_masks, past_kv, x_t, t_tensor)
            x_t = x_t + dt * v_t
        chunks.append(x_t[:, :, :7])  # ВНУТРИ slice :7 — chunk возвращается уже [B, chunk_size, 7]
    
    return torch.stack(chunks, dim=1)  # [B, N, chunk_size, 7]


## 6. Predict functions — plain + PA-RL inference

In [6]:
# === BASELINE inference — fresh SmolVLA без LoRA, без distill checkpoint ===
# Загружается единожды как отдельный policy object — НЕ модифицирует основной policy (distilled)
policy_baseline = None
preprocessor_baseline = None
postprocessor_baseline = None

def load_baseline_policy_lazy():
    """Lazy load — только при необходимости (если RUN_BASELINE_EVAL=True)."""
    global policy_baseline, preprocessor_baseline, postprocessor_baseline
    if policy_baseline is None:
        print(f"Loading BASELINE policy from {MODEL_ID_BASE} (no LoRA)…")
        policy_baseline = SmolVLAPolicy.from_pretrained(MODEL_ID_BASE).to(device)
        preprocessor_baseline, postprocessor_baseline = make_pre_post_processors(
            policy_cfg=policy_baseline.config, pretrained_path=MODEL_ID_BASE,
            preprocessor_overrides={"device_processor": {"device": str(device)}},
        )
        policy_baseline.eval()
        for p in policy_baseline.parameters():
            p.requires_grad_(False)
        print(f"  baseline params: {sum(p.numel() for p in policy_baseline.parameters())/1e6:.1f}M")


@torch.no_grad()
def predict_action_baseline(raw_obs, task_text):
    """Baseline inference — fresh SmolVLA из HuggingFaceVLA/smolvla_libero, no LoRA."""
    obs = libero_obs_to_lerobot(raw_obs, task_text)
    obs = preprocessor_baseline(obs)
    a = policy_baseline.select_action(obs)
    a = postprocessor_baseline(a)
    return a.squeeze(0).detach().cpu().numpy().astype(np.float32)


# === PLAIN inference — IDENTICAL to parl_distill_progressive ===
@torch.no_grad()
def predict_action_plain(raw_obs, task_text):
    obs = libero_obs_to_lerobot(raw_obs, task_text)
    obs = preprocessor(obs)
    a = policy.select_action(obs)
    a = postprocessor(a)
    return a.squeeze(0).detach().cpu().numpy().astype(np.float32)


# === PA-RL inference — IDENTICAL action computation to pa_rl_step in parl_distill ===
# Steps 1-7 из pa_rl_step (без BC distill loss):
# 1. prepare_policy_inputs (тот же путь что sampling в training)
# 2. sample_n_actions → [1, N, chunk_size, 7]
# 3. Critic encode
# 4. Global rerank по Q первого действия → top-M
# 5. Local refine (gripper frozen, sticky mask)
# 6. Eq 4.3 categorical pick
# 7. Target chunk = [a_first, topm[0]_chunk_tail]

_parl_queue = []
_parl_chunk_size = None
_parl_diagnostic_printed = False


def predict_action_parl_reset():
    global _parl_queue, _parl_diagnostic_printed
    _parl_queue = []
    _parl_diagnostic_printed = False


def predict_action_parl(raw_obs, task_text):
    """PA-RL inference action computation — копия pa_rl_step (без BC distill)."""
    global _parl_queue, _parl_chunk_size, _parl_diagnostic_printed
    
    # Re-gen каждые PARL_CHUNK_HORIZON env steps (closed-loop)
    if _parl_chunk_size is not None and len(_parl_queue) <= (_parl_chunk_size - PARL_CHUNK_HORIZON):
        _parl_queue = []
    
    if len(_parl_queue) == 0:
        # ── Получить raw inputs для prepare_policy_inputs (тот же путь что в training pa_rl_step) ──
        a_img = rotate_180(raw_obs["agentview_image"]).astype(np.float32) / 255.0
        w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"]).astype(np.float32) / 255.0
        img1_policy = torch.from_numpy(a_img).permute(2, 0, 1).unsqueeze(0).to(device)
        img2_policy = torch.from_numpy(w_img).permute(2, 0, 1).unsqueeze(0).to(device)
        
        eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
        eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
        grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
        state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
        state_raw = torch.from_numpy(state).unsqueeze(0).to(device)
        
        B = 1
        chunk_size = policy.config.chunk_size
        
        # ── 1. ЕДИНЫЙ путь подготовки inputs (как pa_rl_step) ──
        raw, images, img_masks, state_p, lang_tokens, lang_masks = \
            prepare_policy_inputs(img1_policy, img2_policy, state_raw, B)
        
        # ── 2. Sample N полных chunk-ов в EVAL-режиме ──
        policy.eval()
        candidates_full = sample_n_actions(
            images, img_masks, state_p, lang_tokens, lang_masks,
            n_candidates=PARL_INFERENCE_N,
        )  # [1, N, chunk_size, 7]
        
        # ── 3. Encode obs для critic ──
        img1_critic, img2_critic, state_critic = libero_obs_to_critic_input(raw_obs)
        with torch.no_grad():
            obs_critic = critic.encode(img1_critic, img2_critic, state_critic)  # [1, obs_dim]
        
        # ── 4. Global rerank по Q первого действия (Eq 4.1) ──
        with torch.no_grad():
            c_first = candidates_full[:, :, 0, :].contiguous()  # [1, N, 7]
            obs_exp = obs_critic.unsqueeze(1).expand(-1, PARL_INFERENCE_N, -1)
            q_all = critic.q1(
                obs_exp.reshape(B * PARL_INFERENCE_N, -1),
                c_first.reshape(B * PARL_INFERENCE_N, 7),
            ).reshape(B, PARL_INFERENCE_N)
            topm_q, topm_idx = q_all.topk(PARL_INFERENCE_M, dim=1)
            b_idx = torch.arange(B, device=device).unsqueeze(1).expand(-1, PARL_INFERENCE_M)
            topm_chunks = candidates_full[b_idx, topm_idx]  # [1, M, chunk_size, 7]
            topm_first = topm_chunks[:, :, 0, :].contiguous()  # [1, M, 7]
        
        # ── 5. Local refine: top-M (БЕЗ action_demo, т.к. inference) ──
        # В training: K_init = M_GLOBAL + 1 (top-M + action_demo)
        # В inference: K_init = M (нет action_demo)
        K_init = PARL_INFERENCE_M
        a_flat = topm_first.reshape(B * K_init, 7)
        obs_K = obs_critic.unsqueeze(1).expand(-1, K_init, -1).reshape(B * K_init, -1)
        
        gripper_init = a_flat[:, 6:7].clone()
        a = a_flat.detach().clone()
        sticky = torch.ones(B * K_init, dtype=torch.bool, device=device)
        for _ in range(PARL_INFERENCE_T):
            a_req = a.detach().requires_grad_(True)
            q = critic.q1(obs_K, a_req)
            g = torch.autograd.grad(q.sum(), a_req)[0]
            g[:, 6] = 0.0
            a_new = (a_req + PARL_INFERENCE_ETA * g).clamp(-1, 1).detach()
            a_new[:, 6] = gripper_init.squeeze(-1)
            with torch.no_grad():
                q_new = critic.q1(obs_K, a_new)
                improved = (q_new > q.detach())
                sticky = sticky & improved
                a = torch.where(sticky.unsqueeze(-1), a_new, a)
        
        refined = a.detach().reshape(B, K_init, 7)
        
        # ── 6. Eq 4.3 categorical pick ──
        with torch.no_grad():
            all_q = critic.q1(obs_K, refined.reshape(B * K_init, 7)).reshape(B, K_init)
            q_std = all_q.std(dim=1).mean().item()
            if q_std < 0.1:
                best_idx = all_q.argmax(dim=1)
                a_first = refined[torch.arange(B), best_idx]
            else:
                weights = torch.softmax(all_q, dim=1)
                sampled = torch.multinomial(weights, num_samples=1).squeeze(-1)
                a_first = refined[torch.arange(B), sampled]
            
            if not _parl_diagnostic_printed:
                print(f"      Q stats (after refine): q_min={all_q.min().item():+.2f} q_max={all_q.max().item():+.2f} "
                      f"q_std={q_std:.3f} q_global_spread={q_all.max().item()-q_all.min().item():.3f}", flush=True)
                _parl_diagnostic_printed = True
        
        policy.train()
        
        # ── 7. Target chunk = [a_first, хвост из top-1 sampled chunk-а] ──
        # Same as pa_rl_step
        a_rest = topm_chunks[:, 0, 1:, :].detach()  # [1, chunk_size-1, 7]
        full_chunk = torch.cat([a_first.unsqueeze(1), a_rest], dim=1)  # [1, chunk_size, 7]
        _parl_chunk_size = full_chunk.shape[1]
        
        # Postprocess каждый timestep → env-space
        with torch.no_grad():
            chunk_env = torch.zeros(_parl_chunk_size, 7, device=device)
            for t in range(_parl_chunk_size):
                a_post = postprocessor(full_chunk[0, t:t+1])  # [1, 7] → env-space [1, 7]
                chunk_env[t] = a_post.squeeze(0)
        
        _parl_queue = [chunk_env[t] for t in range(_parl_chunk_size)]
    
    a_env = _parl_queue.pop(0)
    return a_env.detach().cpu().numpy().astype(np.float32)


## 7. Eval function

In [7]:
def eval_libero(predict_fn, label=""):
    """Eval на 13 init_states + per-dim action stats.
    
    IDENTICAL logic to parl_distill_progressive.eval_libero except predict_fn is parameterized.
    """
    policy.eval()
    n_success = 0
    all_actions = []
    t_eval_start = time.time()
    state_results = {}
    
    # Reset PA-RL state если используется PA-RL inference
    if predict_fn is predict_action_parl:
        predict_action_parl_reset()
    
    for state_id in INIT_STATES_IDS:
        try:
            policy.reset()
            # Reset PA-RL queue per state
            if predict_fn is predict_action_parl:
                predict_action_parl_reset()
            
            env.reset()
            raw_obs = env.set_init_state(init_states[state_id])
            success = False
            ep_actions = []
            for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
                if t < NUM_STABILIZATION_STEPS:
                    action = [0.0]*6 + [-1.0]
                else:
                    a = predict_fn(raw_obs, task_description_libero)
                    action = a.tolist()
                    ep_actions.append(a)
                raw_obs, _, done, _ = env.step(action)
                if done:
                    success = True
                    break
            if success: n_success += 1
            all_actions.extend(ep_actions)
            state_results[state_id] = (success, t+1)
            elapsed = time.time() - t_eval_start
            print(f"    state {state_id}: {'OK' if success else '..'} ({t+1} steps, {elapsed:.0f}s)", flush=True)
        except Exception as e:
            print(f"    state {state_id}: ERROR - {type(e).__name__}: {str(e)[:80]}", flush=True)
            state_results[state_id] = (False, 0)
            continue
    
    all_actions = np.array(all_actions) if all_actions else np.zeros((1, 7))
    sr = n_success / len(INIT_STATES_IDS)
    mean_abs = np.abs(all_actions).mean(axis=0) if len(all_actions) else np.zeros(7)
    print(f"  [{label}] SR: {n_success}/13 = {sr:.0%}, action mean abs: {mean_abs.round(3)}", flush=True)
    return n_success, sr, mean_abs, state_results


## 7.5. Baseline eval — multi-seed (fresh SmolVLA, no distill)


In [8]:
baseline_results = []
if RUN_BASELINE_EVAL:
    load_baseline_policy_lazy()
    
    # eval_libero ожидает policy.reset() — у нас есть отдельный policy_baseline
    # Делаем wrapper-функцию eval с baseline policy
    def eval_libero_baseline(predict_fn, label=""):
        n_success = 0
        all_actions = []
        t_eval_start = time.time()
        state_results = {}
        
        for state_id in INIT_STATES_IDS:
            try:
                policy_baseline.reset()
                env.reset()
                raw_obs = env.set_init_state(init_states[state_id])
                success = False
                ep_actions = []
                for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
                    if t < NUM_STABILIZATION_STEPS:
                        action = [0.0]*6 + [-1.0]
                    else:
                        a = predict_fn(raw_obs, task_description_libero)
                        action = a.tolist()
                        ep_actions.append(a)
                    raw_obs, _, done, _ = env.step(action)
                    if done:
                        success = True
                        break
                if success: n_success += 1
                all_actions.extend(ep_actions)
                state_results[state_id] = (success, t+1)
                elapsed = time.time() - t_eval_start
                print(f"    state {state_id}: {'OK' if success else '..'} ({t+1} steps, {elapsed:.0f}s)", flush=True)
            except Exception as e:
                print(f"    state {state_id}: ERROR - {type(e).__name__}: {str(e)[:80]}", flush=True)
                state_results[state_id] = (False, 0)
                continue
        
        all_actions = np.array(all_actions) if all_actions else np.zeros((1, 7))
        sr = n_success / len(INIT_STATES_IDS)
        mean_abs = np.abs(all_actions).mean(axis=0) if len(all_actions) else np.zeros(7)
        print(f"  [{label}] SR: {n_success}/13 = {sr:.0%}, action mean abs: {mean_abs.round(3)}", flush=True)
        return n_success, sr, mean_abs, state_results
    
    print("="*70)
    print(f"BASELINE EVAL — multi-seed (N={len(SEEDS)} seeds × 13 states = {len(SEEDS)*13} episodes)")
    print(f"  Policy: {MODEL_ID_BASE} (no LoRA, no PA-RL distill)")
    print("="*70)
    t0_total = time.time()
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n--- Seed {seed_idx+1}/{len(SEEDS)}: SEED={seed} ---")
        torch.manual_seed(seed)
        np.random.seed(seed)
        env.seed(seed)
        
        t0 = time.time()
        n_succ, sr, mean_abs, state_results = eval_libero_baseline(predict_action_baseline, label=f"BASELINE seed={seed}")
        baseline_results.append((seed, n_succ, sr, mean_abs, state_results))
        print(f"  Time: {(time.time()-t0)/60:.1f} min")
    
    succs = np.array([r[1] for r in baseline_results])
    srs   = np.array([r[2] for r in baseline_results])
    print(f"\n{'─'*50}")
    print(f"BASELINE aggregate:  {succs.mean():.1f} ± {succs.std():.1f} / 13   "
          f"({srs.mean()*100:.1f}% ± {srs.std()*100:.1f}%)")
    print(f"  per-seed: {dict(zip(SEEDS, succs.tolist()))}")
    print(f"  Total time: {(time.time()-t0_total)/60:.1f} min")
else:
    print("[Skipped] RUN_BASELINE_EVAL = False")


[Skipped] RUN_BASELINE_EVAL = False


## 8. Plain eval (baseline reference)

In [9]:
plain_results = []  # list of (n_succ, sr, state_results) per seed
if RUN_PLAIN_EVAL:
    print("="*70)
    print(f"PLAIN EVAL — multi-seed (N={len(SEEDS)} seeds × 13 states = {len(SEEDS)*13} episodes)")
    print("="*70)
    t0_total = time.time()
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n--- Seed {seed_idx+1}/{len(SEEDS)}: SEED={seed} ---")
        torch.manual_seed(seed)
        np.random.seed(seed)
        env.seed(seed)
        
        t0 = time.time()
        n_succ, sr, mean_abs, state_results = eval_libero(predict_action_plain, label=f"PLAIN seed={seed}")
        plain_results.append((seed, n_succ, sr, mean_abs, state_results))
        print(f"  Time: {(time.time()-t0)/60:.1f} min")
    
    # Aggregate stats
    succs = np.array([r[1] for r in plain_results])
    srs   = np.array([r[2] for r in plain_results])
    print(f"\n{'─'*50}")
    print(f"PLAIN aggregate:  {succs.mean():.1f} ± {succs.std():.1f} / 13   "
          f"({srs.mean()*100:.1f}% ± {srs.std()*100:.1f}%)")
    print(f"  per-seed: {dict(zip(SEEDS, succs.tolist()))}")
    print(f"  Total time: {(time.time()-t0_total)/60:.1f} min")
else:
    print("[Skipped] RUN_PLAIN_EVAL = False")


PLAIN EVAL — multi-seed (N=8 seeds × 13 states = 104 episodes)

--- Seed 1/8: SEED=42 ---


    state 1: OK (94 steps, 39s)


    state 2: .. (100 steps, 73s)


    state 6: OK (85 steps, 105s)


    state 7: OK (84 steps, 135s)


...

    state 38: .. (100 steps, 378s)


    state 47: .. (100 steps, 414s)


    state 46: .. (100 steps, 448s)


  [PLAIN seed=34] SR: 5/13 = 38%, action mean abs: [0.433 0.195 0.502 0.028 0.042 0.04  0.998]


  Time: 7.5 min

──────────────────────────────────────────────────
PLAIN aggregate:  8.4 ± 1.6 / 13   (64.4% ± 12.1%)
  per-seed: {42: 8, 228: 10, 282: 9, 1488: 9, 1337: 7, 67: 9, 69: 10, 34: 5}
  Total time: 57.7 min


## 9. PA-RL inference eval (critic-guided)

In [10]:
parl_results = []  # list of (seed, n_succ, sr, mean_abs, state_results)
if RUN_PARL_INFERENCE:
    print("="*70)
    print(f"PA-RL INFERENCE — multi-seed (N={len(SEEDS)} seeds × 13 states = {len(SEEDS)*13} episodes)")
    print(f"  N={PARL_INFERENCE_N}, M={PARL_INFERENCE_M}, T={PARL_INFERENCE_T}, "
          f"η={PARL_INFERENCE_ETA}, horizon={PARL_CHUNK_HORIZON}")
    print("="*70)
    t0_total = time.time()
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n--- Seed {seed_idx+1}/{len(SEEDS)}: SEED={seed} ---")
        torch.manual_seed(seed)
        np.random.seed(seed)
        env.seed(seed)
        
        t0 = time.time()
        n_succ, sr, mean_abs, state_results = eval_libero(predict_action_parl, label=f"PA-RL seed={seed}")
        parl_results.append((seed, n_succ, sr, mean_abs, state_results))
        print(f"  Time: {(time.time()-t0)/60:.1f} min")
    
    # Aggregate stats
    succs = np.array([r[1] for r in parl_results])
    srs   = np.array([r[2] for r in parl_results])
    print(f"\n{'─'*50}")
    print(f"PA-RL aggregate:  {succs.mean():.1f} ± {succs.std():.1f} / 13   "
          f"({srs.mean()*100:.1f}% ± {srs.std()*100:.1f}%)")
    print(f"  per-seed: {dict(zip(SEEDS, succs.tolist()))}")
    print(f"  Total time: {(time.time()-t0_total)/60:.1f} min")
else:
    print("[Skipped] RUN_PARL_INFERENCE = False")


[Skipped] RUN_PARL_INFERENCE = False


## 10. Сравнение результатов

In [11]:
print("="*70)
print(f"MULTI-SEED COMPARISON — Policy: {CHECKPOINT_PATH}")
print(f"  Seeds: {SEEDS}")
print(f"  Total episodes: {len(SEEDS)*13} per mode")
print("="*70)

# Aggregate baseline
if baseline_results:
    base_succs = np.array([r[1] for r in baseline_results])
    base_srs   = np.array([r[2] for r in baseline_results])
    print(f"\nBaseline (SmolVLA, no PA-RL):")
    print(f"  mean = {base_succs.mean():.2f} / 13 = {base_srs.mean()*100:.1f}%")
    print(f"  std  = {base_succs.std():.2f}   ({base_srs.std()*100:.1f}%)")
    print(f"  per-seed:  {dict(zip([r[0] for r in baseline_results], base_succs.tolist()))}")

# Aggregate plain
if plain_results:
    plain_succs = np.array([r[1] for r in plain_results])
    plain_srs   = np.array([r[2] for r in plain_results])
    print(f"\nDistilled (plain inference):")
    print(f"  mean = {plain_succs.mean():.2f} / 13 = {plain_srs.mean()*100:.1f}%")
    print(f"  std  = {plain_succs.std():.2f}   ({plain_srs.std()*100:.1f}%)")
    print(f"  per-seed:  {dict(zip([r[0] for r in plain_results], plain_succs.tolist()))}")

# Aggregate PA-RL
if parl_results:
    parl_succs = np.array([r[1] for r in parl_results])
    parl_srs   = np.array([r[2] for r in parl_results])
    print(f"\nDistilled + PA-RL inference:")
    print(f"  mean = {parl_succs.mean():.2f} / 13 = {parl_srs.mean()*100:.1f}%")
    print(f"  std  = {parl_succs.std():.2f}   ({parl_srs.std()*100:.1f}%)")
    print(f"  per-seed:  {dict(zip([r[0] for r in parl_results], parl_succs.tolist()))}")

# Deltas
def welch_se(a, b):
    return np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))

if baseline_results and plain_results:
    d = plain_succs.mean() - base_succs.mean()
    se = welch_se(base_succs, plain_succs)
    sig = "significant" if abs(d) > 2*se else "within noise"
    print(f"\nΔ (distill − baseline):       {d:+.2f} ep  (SE = {se:.2f}, {sig})")

if plain_results and parl_results:
    d = parl_succs.mean() - plain_succs.mean()
    se = welch_se(plain_succs, parl_succs)
    sig = "significant" if abs(d) > 2*se else "within noise"
    print(f"Δ (PA-RL inf − distill):      {d:+.2f} ep  (SE = {se:.2f}, {sig})")

if baseline_results and parl_results:
    d = parl_succs.mean() - base_succs.mean()
    se = welch_se(base_succs, parl_succs)
    sig = "significant" if abs(d) > 2*se else "within noise"
    print(f"Δ (PA-RL inf − baseline):     {d:+.2f} ep  (SE = {se:.2f}, {sig})")

# Per-state consistency
header = f"\n{'state':<8}"
if baseline_results: header += f" {'baseline':<14}"
if plain_results:    header += f" {'distill':<14}"
if parl_results:     header += f" {'PA-RL inf':<14}"
print(header)
print("-" * len(header))

for state_id in INIT_STATES_IDS:
    line = f"{state_id:<8}"
    if baseline_results:
        b_wins = sum(1 for r in baseline_results if r[4].get(state_id, (False,))[0])
        line += f" {b_wins}/{len(baseline_results):<13}"
    if plain_results:
        p_wins = sum(1 for r in plain_results if r[4].get(state_id, (False,))[0])
        line += f" {p_wins}/{len(plain_results):<13}"
    if parl_results:
        pa_wins = sum(1 for r in parl_results if r[4].get(state_id, (False,))[0])
        line += f" {pa_wins}/{len(parl_results):<13}"
    print(line)

# Report-ready table
print("\n" + "="*70)
print("REPORT TABLE (mean ± std over 3 seeds, 13 states each)")
print("="*70)
if baseline_results:
    print(f"  SmolVLA baseline                  {base_succs.mean():.1f} ± {base_succs.std():.1f} / 13   ({base_srs.mean()*100:.0f}%)")
if plain_results:
    print(f"  + Offline PA-RL distillation      {plain_succs.mean():.1f} ± {plain_succs.std():.1f} / 13   ({plain_srs.mean()*100:.0f}%)")
if parl_results:
    print(f"  + PA-RL inference (critic-guided) {parl_succs.mean():.1f} ± {parl_succs.std():.1f} / 13   ({parl_srs.mean()*100:.0f}%)")


MULTI-SEED COMPARISON — Policy: /workspace/out/smolvla_parl_online_best_12
  Seeds: [42, 228, 282, 1488, 1337, 67, 69, 34]
  Total episodes: 104 per mode

Distilled (plain inference):
  mean = 8.38 / 13 = 64.4%
  std  = 1.58   (12.1%)
  per-seed:  {42: 8, 228: 10, 282: 9, 1488: 9, 1337: 7, 67: 9, 69: 10, 34: 5}

state    distill       
------------------------
1        3/8            
2        6/8            
6        8/8            
7        6/8            
13       2/8            
22       7/8            
23       0/8            
27       7/8            
32       8/8            
35       6/8            
38       5/8            
47       5/8            
46       4/8            

REPORT TABLE (mean ± std over 3 seeds, 13 states each)
  + Offline PA-RL distillation      8.4 ± 1.6 / 13   (64%)
